In [2]:
import keras
import pandas as pd

from fftdeep.data.arrays import get_arrays
from fftdeep.evaluation.metrics import binary_metrics
from fftdeep.paths import PROJECT_ROOT
from fftdeep.training.train_fusion import make_inputs

def load(name):
    return keras.models.load_model(PROJECT_ROOT / "models" / f"cnn_{name}.keras", compile=False)

def evaluate(subset):
    X, y = get_arrays(subset)
    X = X.astype("float32")
    rows = {}
    for name in ["baseline", "deeper_lrdecay"]:
        rows[name] = binary_metrics(y, load(name).predict(X, batch_size=512, verbose=0).ravel())
    for mode in ["image", "freq", "both"]:
        inputs, y2 = make_inputs(mode, subset)
        rows[mode] = binary_metrics(y2, load(mode).predict(inputs, batch_size=512, verbose=0).ravel())
    return pd.DataFrame(rows).T[["accuracy", "precision", "recall", "specificity", "f1", "roc_auc"]].round(4)

val_table = evaluate("val")
val_table.to_csv(PROJECT_ROOT / "reports" / "val_results.csv")
test_table = pd.read_csv(PROJECT_ROOT / "reports" / "test_results.csv", index_col=0)

summary = pd.DataFrame({
    "val_acc": val_table["accuracy"],
    "test_acc": test_table["accuracy"],
    "val_auc": val_table["roc_auc"],
    "test_auc": test_table["roc_auc"],
})
summary["acc_diff"] = (summary["test_acc"] - summary["val_acc"]).round(4)
summary.to_csv(PROJECT_ROOT / "reports" / "val_vs_test.csv")
print(summary)

                val_acc  test_acc  val_auc  test_auc  acc_diff
baseline         0.9474    0.9450   0.9881    0.9884   -0.0024
deeper_lrdecay   0.9619    0.9634   0.9940    0.9938    0.0015
image            0.9494    0.9503   0.9904    0.9910    0.0009
freq             0.8440    0.8420   0.9222    0.9221   -0.0020
both             0.9433    0.9470   0.9872    0.9884    0.0037
